In [ ]:
%%html
<style>
.specta-article-host-widget {
    .specta-cell-output {
        max-width: 900px;
        margin: auto;
    }
}
</style>

In [1]:
from os.path import join
import sys
from glue.core.link_helpers import LinkSame

import numpy as np

from glue.core import Data
from glue.core.subset import AndState, CategorySubsetState, ElementSubsetState, RangeSubsetState
from glue.core.subset_group import GroupedSubset

from glue_jupyter import jglue
from glue_jupyter.bqplot.histogram import BqplotHistogramView
from glue_jupyter.bqplot.scatter import BqplotScatterView
from glue_jupyter.table.viewer import TableViewer

In [2]:
from os.path import splitext
import string

from glue.core.link_helpers import LinkSame


def printable_string(s):
    return "".join(filter(lambda x: x in string.printable, s))


# Some of the components have non-printable characters in their names
# when loaded from the CSV, which makes them hard to grab later.
# So we strip any non-printable characters from the component names
def clean_component_names(data):
    for component in data.components:
        component.label = printable_string(component.label)


def layer_for_data(viewer, data):
    for layer in viewer.layers:
        if layer.layer is data:
            return layer
    return None


# TODO: This is not very robust, just to get things working
def color_for_path(filepath, colors):
    name, ext = splitext(filepath)
    if ext != ".csv":
        return None 
    for color in colors:
        color = str(color)
        if color.lower() in name.lower():
            return color

    return None

def color_filename(color):
    return f"GenEd 1112 {color.upper()}.csv"

In [3]:
app = jglue()
dc = app.data_collection

/opt/homebrew/Caskroom/mambaforge/base/envs/glue/lib/python3.11/site-packages/glue_plotly/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/Users/jon/dev/glue/glue-qt/glue_qt/viewers/common/tool.py:5: UserWarning:

glue.viewers.common.tool is deprecated, use glue.viewers.common.tool instead



In [4]:
data_filepath_2025 = "2025_student_results.csv"
data = app.load_data(data_filepath_2025)
clean_component_names(data)
data.style.color = "#ffffff"
colors = np.unique(data["Color"])
for color, code in zip(colors, colors.codes) :
    dc.new_subset_group(
        label=color,
        subset_state=CategorySubsetState(att=data.id["Color"], categories=[code]),
        color=color,
        alpha=1
    )

In [5]:
# Make the "1-1" data
# and link it to our main data set
one_to_one = Data(label="one-to-one", Uncertainty=[0, 1000], End=[0, 1000])
dc.append(one_to_one)
dc.add_link(LinkSame(data.id["UNCERTAINTY"], one_to_one.id["Uncertainty"]))
dc.add_link(LinkSame(data.id["End_off_feet"], one_to_one.id["End"]))

In [6]:
# Find the winner(s) and create a subset
min_distance = min(data["End_off_feet"])
winner_sg = dc.new_subset_group(
    label="Winners 2025",
    subset_state=RangeSubsetState(lo=min_distance, hi=min_distance,
                                 att=data.id["End_off_feet"]),
    color="Purple",
    alpha=1
)

In [7]:
# Create a subset for "Student Picker"
# JC: I don't know what this means
student_picker_sg = dc.new_subset_group(
    label="Student Picker",
    subset_state=ElementSubsetState(indices=[37], data=data),
    color="#941751",
    alpha=1
)

In [8]:
# Make a subset for the correct end
lat = data["End_Lat_true"][0]
lon = data["End_Long_true"][0]
lat_ss = RangeSubsetState(lo=lat, hi=lat, att=data.id["EndLat"])
lon_ss = RangeSubsetState(lo=lon, hi=lon, att=data.id["EndLong"])
endpoint_sg = dc.new_subset_group(label="Common End",
                                    subset_state=AndState(lat_ss, lon_ss),
                                    color="Red")

In [9]:
locations_scatter = app.new_data_viewer(BqplotScatterView, data=data)
locations_scatter.state.x_att = data.id["EndLong"]
locations_scatter.state.y_att = data.id["EndLat"]
for layer in locations_scatter.layers:
    if layer.layer.label == "Common End":
        layer.state.size = 12
        layer.state.alpha = 1

LayoutWidget(controls={'toolbar_selection_tools': BasicJupyterToolbar(template=Template(template='<template>\n…

In [10]:
stats_histogram = app.new_data_viewer(BqplotHistogramView, data=data)
stats_histogram.state.x_att = data.id["Color"]
stats_histogram.hist_n_bins = 5

hide_layers = ("2025_student_results", "Winners 2025", "Common End", "Student Picker")
for layer in stats_histogram.layers:
    if layer.layer.label in hide_layers:
        layer.state.visible = False

LayoutWidget(controls={'toolbar_selection_tools': BasicJupyterToolbar(template=Template(template='<template>\n…

In [11]:
# The bounds of this graph are really wonky because someone put 
# in 71.1173 instead of -71.117 for PointBLong
# You can ignore this outlier by using the commented-out lines below
b_scatter = app.new_data_viewer(BqplotScatterView, data=data)
b_scatter.state.x_att = data.id["PointBLong"]
b_scatter.state.y_att = data.id["PointBLat"]
# b_scatter.state.x_min = -71.119
# b_scatter.state.x_max = -71.116

for layer in b_scatter.layers:
    if layer.layer.label == "Student Picker":
        layer.state.visible = False
        break

LayoutWidget(controls={'toolbar_selection_tools': BasicJupyterToolbar(template=Template(template='<template>\n…

In [12]:
line_scatter = app.new_data_viewer(BqplotScatterView, data=data)
line_scatter.state.x_att = data.id["UNCERTAINTY"]
line_scatter.state.y_att = data.id["End_off_feet"]
line_scatter.add_data(one_to_one)

one_to_one_layer = layer_for_data(line_scatter, one_to_one)
one_to_one_layer.state.line_visible = True
one_to_one_layer.state.linewidth = 3
one_to_one_layer.state.linestyle = "dashed"

winners_layer = None
student_picker_layer = None
for layer in line_scatter.layers:
    if isinstance(layer.layer, GroupedSubset) and layer.layer.data == data:
        if layer.layer.label == "Winners 2025":
            winners_layer = layer
        elif layer.layer.label == "Student Picker":
            student_picker_layer = layer
        if winners_layer and student_picker_layer:
            break

winners_layer.state.size = 200
winners_layer.state.fill = False

student_picker_layer.state.size = 500

LayoutWidget(controls={'toolbar_selection_tools': BasicJupyterToolbar(template=Template(template='<template>\n…

In [13]:
# There seem to be issues with a JSON serialization step for the table viewer for integers
# so we change the values in the `number` component to be floats
data.add_component(data["number"].astype(np.float64), label="number")
table = app.new_data_viewer(TableViewer, data=data)

/opt/homebrew/Caskroom/mambaforge/base/envs/glue/lib/python3.11/site-packages/jupyter_client/session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant



LayoutWidget(controls={'toolbar_selection_tools': BasicJupyterToolbar(template=Template(template='<template>\n…

In [14]:
from pywwt.jupyter import WWTJupyterWidget
from glue_wwt.viewer.tools import RefreshTileCacheTool
from glue_wwt.viewer.jupyter.viewer import WWTJupyterViewer

class GlueLiteWWTViewer(WWTJupyterViewer):

    def _initialize_wwt(self):
        self._wwt = WWTJupyterWidget(app_url="https://web.wwtassets.org/research/latest/")

wwt = app.new_data_viewer(GlueLiteWWTViewer, data=data)
wwt.state.mode = "Earth"
wwt.state.lon_att = data.id["EndLong"]
wwt.state.lat_att = data.id["EndLat"]